# Sequence evolution and variant fitness in Covasim

This notebook demonstrates two linked features:

1. **Sequence evolution** — Each transmission accumulates nucleotide mutations sampled from a Jukes-Cantor (JC69) model. `infection_log` entries carry `branch_length`, `branch_mutations`, and `n_mutations`; full haplotypes are reconstructed on demand via `sequence_tracker.reconstruct_haplotype()`.

2. **Variant fitness** — `BloomNtFitnessModel` (Bloom lab `nt_fitness.csv`) maps each haplotype's accumulated mutations to a log-fitness score and multiplies the agent's transmissibility by `exp(Σ fitness_i × scale)`. Fitter haplotypes spread faster with no manual `rel_beta` tuning required.

**Requirements:**
- `NC_045512_Hu-1.fasta` — Wuhan-1 reference genome (29 903 nt)
- `data/nt_fitness.csv` — Bloom lab nucleotide fitness table
- `ARTIC_V4-1.bed` — primer BED for wastewater read simulation (Part 4)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import covasim as cv

# Paths are relative to docs/tutorials/ — run the notebook from that directory.
REF_FASTA     = 'sars-2-data/NC_045512_Hu-1.fasta'
AY4_FASTA     = 'sars-2-data/variants/AY.4_consensus.fasta'
ARTIC_PRIMERS = 'sars-2-data/ARTIC_V4-1.bed'

## Part 1: Basic sequence evolution

The minimal config is `evo_pars['enable'] = True` plus a reference FASTA. The JC69 model draws `Poisson(L × μ × Δt)` substitutions per transmission edge, where `Δt` is the gap in days between source and target exposure.

In [ ]:
sim1 = cv.Sim(dict(
    pop_size     = 50_000,
    pop_infected = 10, # Several seed infections, since an outbreak from a single one often dies out
    start_day    = '2020-01-01',
    end_day      = '2020-06-01',
    evo_pars = dict(
        enable            = True,
        reference         = REF_FASTA,
        mol_clock_rate    = 1e-6,   # ~1 substitution/genome/month at 30 000 nt
        sub_model         = 'JC',
    ),
))
sim1.run()

In [ ]:
log    = sim1.people.infection_log
n_muts = [e['n_mutations'] for e in log if 'n_mutations' in e]

print(f'Transmission events:    {len(log):,}')
print(f'Mean branch mutations:  {np.mean(n_muts):.3f}')
print(f'Max branch mutations:   {max(n_muts)}')

In [ ]:
# Distribution of branch mutations across all transmission edges
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(n_muts, bins=range(max(n_muts) + 2), color='steelblue', edgecolor='white')
ax.set_xlabel('Branch mutations per transmission')
ax.set_ylabel('Count')
ax.set_title('JC69 branch mutation distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Reconstruct haplotypes for the first few agents and count divergence from reference
tracker = sim1.sequence_tracker
ref_seq = cv.decode_sequence(tracker.reference)

sample_agents = [int(e['target']) for e in log[:5]]
for agent in sample_agents:
    hap   = tracker.reconstruct_haplotype(agent)
    ndiff = sum(a != b for a, b in zip(hap, ref_seq))
    print(f'  Agent {agent:6d}: {ndiff} mutation(s) vs reference')

## Part 2: Fitness-weighted sequence evolution

`BloomNtFitnessModel` looks up each mutation in the Bloom lab table and computes:

$$\text{rel\_beta} \leftarrow \exp\!\left(\sum_i \text{fitness}_i \times \text{scale}\right)$$

Beneficial mutations (positive log-fitness) propagate faster; deleterious ones are suppressed — driven entirely by empirical data, not hand-tuned parameters.

In [ ]:
sim2 = cv.Sim(dict(
    pop_size             = 50_000,
    pop_infected         = 10, # Several seed infections, since an outbreak from a single one often dies out
    start_day            = '2020-01-01',
    end_day              = '2020-06-01',
    evo_pars = dict(
        enable            = True,
        reference         = REF_FASTA,
        mol_clock_rate    = 1e-6,
        sub_model         = 'JC',
        fitness_model     = 'bloom_nt',
        fitness_scale     = 0.1,   # tune to calibrate how strongly fitness shapes sweeps
    ),
))
sim2.run()

In [ ]:
tracker2    = sim2.sequence_tracker
all_agents  = [int(e['target']) for e in sim2.people.infection_log]
multipliers = [tracker2.get_fitness_multiplier(a) for a in all_agents]

print(f'Mean fitness multiplier:   {np.mean(multipliers):.4f}')
print(f'Max fitness multiplier:    {max(multipliers):.4f}')
print(f'Agents with rel_beta > 1:  {sum(m > 1.0 for m in multipliers):,}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(multipliers, bins=50, color='darkorange', edgecolor='white')
ax.axvline(1.0, color='black', linestyle='--', linewidth=1, label='neutral (1.0)')
ax.set_xlabel('Fitness multiplier (rel_beta)')
ax.set_ylabel('Agents')
ax.set_title('Haplotype fitness distribution across infected agents')
ax.legend()
plt.tight_layout()
plt.show()

## Part 3: Variant competition with a founding genotype

When `cv.variant()` is given a `founding_fasta`, the tracker loads that consensus sequence, diffs it against the reference, and seeds every import of that variant with its characteristic mutation profile. Subsequent evolution accumulates on top.

Here AY.4 (Delta) arrives on day 20. Its Bloom fitness scores from its founding mutations shape the competition against the wild-type.

In [ ]:
ay4 = cv.variant(
    variant = {},
    label     = 'AY.4',
    days      = 20,
    n_imports = 30,
    founding_fasta = AY4_FASTA,
)

sim3 = cv.Sim(
    dict(
        pop_size     = 50_000,
        pop_infected = 20,
        start_day    = '2022-10-01',
        end_day      = '2023-04-01',
        evo_pars = dict(
            enable            = True,
            reference         = REF_FASTA,
            mol_clock_rate    = 1e-6,
            sub_model         = 'JC',
            fitness_model     = 'bloom_nt',
                fitness_scale     = 0.1,
        ),
    ),
    variants  = [ay4],
    analyzers = [cv.WastewaterSampler(days=list(range(0, 180, 14)))],
)
sim3.run()

In [ ]:
# Cumulative infections by variant (columns: 0 = wild-type, 1 = AY.4)
cumulative = sim3.results['variant']['cum_infections_by_variant'].values
print(f'Wild-type cumulative infections:  {int(cumulative[-1, 0]):,}')
print(f'AY.4 cumulative infections:       {int(cumulative[-1, 1]):,}')

In [ ]:
# Variant prevalence over time
prev = sim3.results['variant']['prevalence_by_variant'].values.T # (variant, day)
tvec = sim3.results['t']

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tvec, prev[0], label='Wild-type', color='steelblue')
ax.plot(tvec, prev[1], label='AY.4',      color='darkorange')
ax.axvline(20, color='gray', linestyle=':', linewidth=1, label='AY.4 introduction (day 20)')
ax.set_xlabel('Day')
ax.set_ylabel('Prevalence')
ax.set_title('Variant competition: wild-type vs AY.4 (Delta)')
ax.legend()
plt.tight_layout()
plt.show()